In [1]:
!pip install ultralytics kaggle --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.5 MB/s eta 0:00:00


In [2]:
!mkdir -p ~/.kaggle

In [3]:
!mv /content/kaggle.json ~/.kaggle/

In [4]:
!chmod 600 ~/.kaggle/kaggle.json

In [6]:
!kaggle datasets download lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes

Dataset URL: https://www.kaggle.com/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes
License(s): MIT
 82% 152M/185M [00:00<00:00, 1.59GB/s]
100% 185M/185M [00:00<00:00, 1.08GB/s]


In [7]:
!unzip -q road-damage-dataset-potholes-cracks-and-manholes.zip -d road_damage

In [9]:
!ls road_damage/data

annotations_coco.json	   images  labels-YOLO	YOLO-conversion-script.py
COCO-conversion-script.py  labels  README.md


In [11]:
!ls road_damage/data/images | head

20250216_164325.jpg
20250216_164521.jpg
20250216_164541.jpg
20250219_164649.jpg
20250219_164714.jpg
20250219_164738.jpg
20250219_164746.jpg
20250219_164754.jpg
20250219_164756.jpg
20250219_164807.jpg


In [12]:
#Making train and test folders
import os
import random
import shutil

random.seed(42)

IMG_DIR = "road_damage/data/images"
LBL_DIR = "road_damage/data/labels-YOLO"

# create target dirs
os.makedirs("road_damage/data/images/train", exist_ok=True)
os.makedirs("road_damage/data/images/val", exist_ok=True)
os.makedirs("road_damage/data/labels/train", exist_ok=True)
os.makedirs("road_damage/data/labels/val", exist_ok=True)

images = [f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.png'))]
images.sort()
random.shuffle(images)

split_idx = int(0.8 * len(images))
train_imgs = images[:split_idx]
val_imgs = images[split_idx:]

def move_pair(img_list, split):
    for img in img_list:
        base = os.path.splitext(img)[0]
        lbl = base + ".txt"

        shutil.move(f"{IMG_DIR}/{img}", f"{IMG_DIR}/{split}/{img}")
        shutil.move(f"{LBL_DIR}/{lbl}", f"road_damage/data/labels/{split}/{lbl}")

move_pair(train_imgs, "train")
move_pair(val_imgs, "val")

print("Train images:", len(train_imgs))
print("Val images:", len(val_imgs))

Train images: 1607
Val images: 402


In [13]:
!ls road_damage/data/labels/train | head

20250216_164325.txt
20250216_164541.txt
20250219_164649.txt
20250219_164738.txt
20250219_164754.txt
20250219_164756.txt
20250219_164807.txt
20250219_164813.txt
20250219_164823.txt
20250219_164829.txt


In [19]:
#YOLO config
%%bash
cat > road_damage.yaml << 'EOF'
path: road_damage/data
train: images/train
val: images/val

names:
  0: pothole
  1: crack
  2: manhole
EOF

In [21]:
!yolo val model=yolov8n.yaml data=road_damage.yaml imgsz=640 batch=1

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 73 layers, 3,151,904 parameters, 31,920 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2054.7±364.8 MB/s, size: 104.3 KB)
val: Scanning /content/road_damage/data/labels/val.cache... 402 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 402/402 60.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 402/402 82.1it/s 4.9s
                   all        402       1008          0          0          0          0
                     0        162        263          0          0          0          0
                     1        291        558          0          0          0          0
                     2        140        187          0          0          0          0
Speed: 0.8ms preprocess, 7.9ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /content/runs/detect/

In [22]:
!yolo train model=yolov8n.yaml data=road_damage.yaml imgsz=640 batch=16 epochs=100

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=road_damage.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plot

In [23]:
!yolo predict \
  model=runs/detect/train/weights/best.pt \
  source=road_damage/data/images/val \
  imgsz=640 \
  conf=0.4 \
  save=True

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs

image 1/402 /content/road_damage/data/images/val/20250216_164521.jpg: 384x640 1 manhole, 47.5ms
image 2/402 /content/road_damage/data/images/val/20250219_164714.jpg: 384x640 (no detections), 6.3ms
image 3/402 /content/road_damage/data/images/val/20250219_164746.jpg: 384x640 (no detections), 5.6ms
image 4/402 /content/road_damage/data/images/val/20250219_164838.jpg: 384x640 1 pothole, 5 cracks, 1 manhole, 5.4ms
image 5/402 /content/road_damage/data/images/val/20250219_164848.jpg: 384x640 (no detections), 5.5ms
image 6/402 /content/road_damage/data/images/val/20250219_165056.jpg: 384x640 (no detections), 5.5ms
image 7/402 /content/road_damage/data/images/val/20250219_165351.jpg: 384x640 (no detections), 5.5ms
image 8/402 /content/road_damage/data/images/val/20250219_165753.jpg: 384x640 (no detections), 5.5ms
image 9/402 /con

In [ ]:
# =============================================================================
# ROAD DAMAGE DETECTION USING YOLOv8 — COMPLETE END-TO-END NOTES
# =============================================================================
#
# This notebook documents, in extreme detail, the complete process of building
# a real-world object detection system for detecting road defects using YOLOv8.
#
# This documentation is intentionally written assuming:
#   - ZERO prior knowledge of YOLO
#   - ZERO prior knowledge of object detection
#   - ZERO prior knowledge of datasets, annotations, or training pipelines
#
# If you read this entire comment block carefully, you should be able to:
#   - Understand what object detection is
#   - Understand how YOLO works internally (conceptually)
#   - Understand how datasets are annotated
#   - Understand how training and evaluation work
#   - Rebuild this project from scratch months or years later
#
# =============================================================================
# SECTION 1 — WHAT PROBLEM ARE WE SOLVING?
# =============================================================================
#
# We are solving an OBJECT DETECTION problem.
#
# Object detection means:
#   - Detecting WHERE an object is in an image
#   - AND identifying WHAT the object is
#
# This is different from:
#
# 1. Image classification:
#    - One label for the whole image
#    - Example: "This image contains a pothole"
#
# 2. Object detection:
#    - Multiple objects per image
#    - Each object has a bounding box + label
#    - Example: "This image contains 2 potholes and 1 crack, here are their locations"
#
# In this project, the objects are ROAD DEFECTS:
#   - Potholes
#   - Cracks
#   - Manholes
#
# The output of the model is:
#   - Bounding boxes around defects
#   - Class label for each bounding box
#
# =============================================================================
# SECTION 2 — WHAT IS YOLO?
# =============================================================================
#
# YOLO stands for "You Only Look Once".
#
# YOLO is a family of object detection algorithms that work as follows:
#
#   - The entire image is passed through a neural network ONE TIME
#   - The network directly predicts:
#       * Bounding box coordinates
#       * Class probabilities
#       * Confidence scores
#
# This is why it is fast.
#
# Older methods (R-CNN family):
#   - Generate thousands of region proposals
#   - Run classification on each region
#   - Very slow
#
# YOLO:
#   - Single forward pass
#   - End-to-end learning
#   - Suitable for real-time applications (video, dashcams, surveillance)
#
# =============================================================================
# SECTION 3 — WHAT IS YOLOv8?
# =============================================================================
#
# YOLOv8 is a modern implementation of YOLO maintained by Ultralytics.
#
# Key characteristics of YOLOv8:
#   - Anchor-free detection (simpler and more stable)
#   - CNN-based backbone and detection head
#   - Modular and configurable via YAML files
#
# YOLOv8 supports multiple tasks:
#   - Classification
#   - Detection   ← USED IN THIS PROJECT
#   - Segmentation
#   - Pose estimation
#   - Tracking
#
# =============================================================================
# SECTION 4 — DATASET USED
# =============================================================================
#
# Dataset:
#   "Road Damage Dataset: Potholes, Cracks and Manholes"
#
# Source:
#   Kaggle
#
# Why this dataset?
#   - Real road images (not synthetic)
#   - Bounding-box annotations available
#   - Multiple damage classes
#   - Suitable size for experimentation
#
# =============================================================================
# SECTION 5 — HOW ARE IMAGES ANNOTATED?
# =============================================================================
#
# Object detection models need GROUND TRUTH labels.
#
# For YOLO, each image has a corresponding .txt file that contains annotations.
#
# Example:
#   image: 20250216_164325.jpg
#   label: 20250216_164325.txt
#
# Each line in the label file corresponds to ONE object in the image.
#
# =============================================================================
# SECTION 6 — YOLO ANNOTATION FORMAT (VERY IMPORTANT)
# =============================================================================
#
# Each line in a YOLO label file looks like this:
#
#   class_id x_center y_center width height
#
# ALL values except class_id are NORMALIZED between 0 and 1.
#
# Let’s explain each term in detail.
#
# -----------------------------------------------------------------------------
# 1. class_id
# -----------------------------------------------------------------------------
#
# This is an integer representing the object class.
#
# In this project:
#   0 → pothole
#   1 → crack
#   2 → manhole
#
# Example:
#   0 means the object is a pothole
#
# -----------------------------------------------------------------------------
# 2. x_center
# -----------------------------------------------------------------------------
#
# x_center is the X-coordinate of the CENTER of the bounding box.
#
# IMPORTANT:
#   - It is NOT in pixels
#   - It is a fraction of the image width
#
# Example:
#   x_center = 0.5
#
# Means:
#   The center of the bounding box is at 50% of the image width
#
# If image width = 1000 pixels:
#   x_center = 0.5 → x = 500 pixels
#
# -----------------------------------------------------------------------------
# 3. y_center
# -----------------------------------------------------------------------------
#
# y_center is the Y-coordinate of the CENTER of the bounding box.
#
# Again:
#   - NOT pixels
#   - Fraction of image height
#
# Example:
#   y_center = 0.25
#
# Means:
#   The center of the box is at 25% of image height from the top
#
# -----------------------------------------------------------------------------
# 4. width
# -----------------------------------------------------------------------------
#
# width is the WIDTH of the bounding box,
# expressed as a fraction of the total image width.
#
# Example:
#   width = 0.2
#
# Means:
#   The bounding box is 20% as wide as the image
#
# -----------------------------------------------------------------------------
# 5. height
# -----------------------------------------------------------------------------
#
# height is the HEIGHT of the bounding box,
# expressed as a fraction of the total image height.
#
# Example:
#   height = 0.1
#
# Means:
#   The bounding box height is 10% of the image height
#
# -----------------------------------------------------------------------------
# FULL EXAMPLE
# -----------------------------------------------------------------------------
#
# Line in label file:
#   0 0.52 0.63 0.21 0.18
#
# Interpretation:
#   - Object class: pothole
#   - Center X at 52% of image width
#   - Center Y at 63% of image height
#   - Box width = 21% of image width
#   - Box height = 18% of image height
#
# =============================================================================
# SECTION 7 — WHY NORMALIZED COORDINATES?
# =============================================================================
#
# Normalization makes the model:
#   - Independent of image resolution
#   - Able to generalize across different image sizes
#
# YOLO internally converts these normalized values back to pixels during training
# and inference.
#
# =============================================================================
# SECTION 8 — DATASET STRUCTURE REQUIRED BY YOLO
# =============================================================================
#
# YOLO expects the following directory structure:
#
#   dataset_root/
#     images/
#       train/
#       val/
#     labels/
#       train/
#       val/
#
# Each image in images/train must have a matching label file in labels/train.
#
# Same for validation.
#
# =============================================================================
# SECTION 9 — TRAIN / VALIDATION SPLIT
# =============================================================================
#
# Training set:
#   - Used to UPDATE model weights
#
# Validation set:
#   - Used to EVALUATE model performance
#   - Model NEVER trains on validation images
#
# In this project:
#   - 80% images → training
#   - 20% images → validation
#
# =============================================================================
# SECTION 10 — WHAT IS road_damage.yaml?
# =============================================================================
#
# YOLO does NOT guess dataset details.
#
# We must explicitly tell it:
#   - Where images are
#   - Where labels are
#   - What class IDs mean
#
# road_damage.yaml does exactly that.
#
# =============================================================================
# SECTION 11 — PRETRAINED VS FROM SCRATCH
# =============================================================================
#
# Pretrained model (.pt):
#   - Learned from COCO dataset
#   - Faster convergence
#   - Hidden biases from COCO classes
#
# From scratch (.yaml):
#   - Random initialization
#   - No external knowledge
#   - Slower learning
#   - Academically clean
#
# This project intentionally trains FROM SCRATCH.
#
# =============================================================================
# SECTION 12 — WHY USE yolov8n.yaml?
# =============================================================================
#
# yolov8n.yaml contains:
#   - Model architecture only
#   - No weights
#
# Using this ensures:
#   - Fresh detection head
#   - Correct number of classes
#   - No COCO class leakage
#
# =============================================================================
# SECTION 13 — VALIDATION BEFORE TRAINING
# =============================================================================
#
# We validate BEFORE training to:
#   - Ensure dataset loads
#   - Ensure labels parse correctly
#   - Ensure class mapping is correct
#
# Metrics will be zero at this stage.
# This is EXPECTED.
#
# =============================================================================
# SECTION 14 — TRAINING
# =============================================================================
#
# Training command:
#
#   yolo train model=yolov8n.yaml data=road_damage.yaml
#
# YOLO learns:
#   - Bounding box regression
#   - Classification
#   - Object confidence
#
# =============================================================================
# SECTION 15 — MODEL OUTPUTS
# =============================================================================
#
# YOLO automatically saves:
#   - Best model
#   - Loss curves
#   - Confusion matrix
#
# These are saved under:
#   runs/detect/train/
#
# =============================================================================
# SECTION 16 — MODEL TESTING
# =============================================================================
#
# We test the trained model using:
#
#   yolo predict model=best.pt source=val_images
#
# This produces visual results with bounding boxes drawn.
#
# =============================================================================
# SECTION 17 — MODEL EVALUATION METRICS (DETAILED)
# =============================================================================
#
# After training, we must quantify how good the model actually is.
# Object detection evaluation is more complex than classification.
#
# -----------------------------------------------------------------------------
# CORE CONCEPTS: TP, FP, FN
# -----------------------------------------------------------------------------
#
# TRUE POSITIVE (TP):
#   - Model predicts a bounding box
#   - Predicted class is correct
#   - Bounding box overlaps sufficiently with ground truth
#
# FALSE POSITIVE (FP):
#   - Model predicts a bounding box
#   - But no matching ground-truth object exists
#   - OR the predicted class is wrong
#
# FALSE NEGATIVE (FN):
#   - A ground-truth object exists
#   - But the model fails to detect it
#
# -----------------------------------------------------------------------------
# IOU (INTERSECTION OVER UNION)
# -----------------------------------------------------------------------------
#
# IOU measures how much a predicted bounding box overlaps
# with the ground-truth bounding box.
#
# IOU = (Area of Overlap) / (Area of Union)
#
# IOU ranges from 0 to 1.
# A common threshold is IOU ≥ 0.5.
#
# -----------------------------------------------------------------------------
# PRECISION
# -----------------------------------------------------------------------------
#
# Precision answers:
#   "Of all predicted objects, how many were correct?"
#
# Precision = TP / (TP + FP)
#
# High precision means:
#   - Few false alarms
#
# -----------------------------------------------------------------------------
# RECALL
# -----------------------------------------------------------------------------
#
# Recall answers:
#   "Of all real objects, how many were detected?"
#
# Recall = TP / (TP + FN)
#
# High recall means:
#   - Few missed objects
#
# -----------------------------------------------------------------------------
# PRECISION–RECALL TRADE-OFF
# -----------------------------------------------------------------------------
#
# Increasing confidence threshold:
#   ↑ Precision, ↓ Recall
#
# Decreasing confidence threshold:
#   ↓ Precision, ↑ Recall
#
# -----------------------------------------------------------------------------
# AVERAGE PRECISION (AP)
# -----------------------------------------------------------------------------
#
# AP summarizes the precision–recall curve into a single number
# by integrating precision over recall.
#
# AP is computed per class.
#
# -----------------------------------------------------------------------------
# mAP (MEAN AVERAGE PRECISION)
# -----------------------------------------------------------------------------
#
# mAP is the mean of AP values across all classes.
#
# -----------------------------------------------------------------------------
# mAP@0.5
# -----------------------------------------------------------------------------
#
# mAP@0.5 counts a detection as correct if IOU ≥ 0.5.
# It emphasizes correct detection over perfect localization.
#
# -----------------------------------------------------------------------------
# mAP@0.5:0.95
# -----------------------------------------------------------------------------
#
# This is the COCO-style metric.
# mAP is averaged over IOU thresholds from 0.5 to 0.95.
#
# This metric penalizes poor localization more heavily
# and is commonly used in research papers.
#
# =============================================================================
# SECTION 18 — FINAL REMARKS
# =============================================================================
#
# This notebook documents a FULL OBJECT DETECTION PIPELINE:
#   - Dataset understanding
#   - Annotation format
#   - Training from scratch
#   - Evaluation
#   - Inference
#
# This knowledge transfers directly to:
#   - Autonomous driving
#   - Surveillance
#   - Medical imaging
#   - Robotics
#
# =============================================================================
# END OF DOCUMENTATION
# =============================================================================